[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/07_aggregation_trends.ipynb)

# Sitzung 7 — Aggregation & Trends

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Ihr könnt Bewertungen einordnen und Themen erkennen. Heute die Frage, die eine **Entscheiderin** wirklich stellt: *Wird es besser oder schlechter? Worüber wird am meisten geschimpft? Hat das Update im Herbst geholfen?* Aus Einzelurteilen wird die **analytische Ebene**.

## 0. Setup, Daten, Klassifikation

In [ ]:
import random, re
from collections import Counter, defaultdict
from datetime import date, timedelta
import matplotlib.pyplot as plt
print('Fertig.')

In [ ]:
import random
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"

POS = ["der Klang ist hervorragend", "satte Bässe", "die Geräuschunterdrückung ist top",
       "der Akku hält den ganzen Tag", "sitzt super bequem", "Bluetooth verbindet sofort",
       "top verarbeitet", "klasse für den Preis", "die App ist übersichtlich",
       "die Passform ist perfekt", "der Sound ist klar und ausgewogen"]
NEG = ["die App stürzt ständig ab", "der rechte Ohrhörer lädt nicht mehr",
       "die Geräuschunterdrückung rauscht", "viel zu teuer", "die Touch-Steuerung reagiert kaum",
       "das Case wirkt billig", "der Akku ist nach einer Stunde leer",
       "die Verbindung bricht ab", "sie fallen leicht aus dem Ohr", "der Bass ist matschig"]

POS_OPENERS = ["Bin begeistert:", "Wirklich gut:", "Kann ich empfehlen –", "Top Kauf.",
               "Sehr zufrieden:", "Absolute Kaufempfehlung.", "Ich liebe sie:",
               "Klare Sache:", "Rundum gelungen:", "Volle Punktzahl:", "Endlich zufrieden:",
               "Was soll ich sagen –", "Genau richtig:", "Bestellung hat sich gelohnt:"]
NEG_OPENERS = ["Enttäuschend:", "Leider schlecht:", "Finger weg –", "Bin frustriert:",
               "Nicht zu empfehlen.", "Schade um das Geld:", "Ärgerlich:",
               "Reklamiert:", "Bin raus:", "Nie wieder:", "Herbe Enttäuschung:",
               "Das war nichts:", "Zurückgeschickt:", "Vorsicht:"]
NEU_TEMPLATES = ["Ganz okay, {a}, aber nichts Besonderes.",
                 "Erfüllt seinen Zweck. {a_cap}.",
                 "Durchschnittlich. {a_cap}, mehr nicht.",
                 "Habe sie seit Kurzem, {a} – kann noch nicht viel sagen."]
SARCASTIC = ["Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
             "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
             "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
             "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
             "Klasse, nach einer Woche nur noch Rauschen. Wirklich durchdacht.",
             "Perfekt, der linke fällt ständig raus. Genau mein Wunsch.",
             "Herrlich, die Verbindung bricht alle fünf Minuten ab. Danke auch.",
             "Sensationell, das Case bricht beim ersten Öffnen. Qualität eben.",
             "Bravo, nach dem Update ist die Hälfte der Funktionen weg.",
             "Fantastisch leise – weil nach zwei Tagen einfach tot."]
FAKE = ["BESTES PRODUKT EVER!!! Kauft bei www.super-deals-guenstig.example!!!",
        "5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
        "Gutschein Code NIMBUS100 auf meiner Seite jetzt klicken!!!",
        "amazing product best quality buy now discount link in profile",
        "TOP TOP TOP unbedingt kaufen billigster preis hier klicken",
        "gratis versand nur heute!!! rabattcode DEAL22 einlösen!!!",
        "beste kopfhoerer der welt jetzt zuschlagen link im profil",
        "WOW einfach WOW kaufen kaufen kaufen bester preis garantiert",
        "unglaublich guenstig hier klicken und sparen sparen sparen",
        "mega angebot heute -70% nur ueber meinen link!!!"]
ENGLISH = [("Sound quality is great but the app is a disaster.", "mixed"),
           ("Battery life is amazing, best earbuds I have owned.", "positive"),
           ("Stopped working after a week, very disappointed.", "negative"),
           ("Comfortable fit and clear sound, happy with the purchase.", "positive"),
           ("The noise cancelling is weak and the case feels cheap.", "negative")]
JUNK = ["", "   ", ".", "???", "kein kommentar", "-", "n/a", "...", "!!", "??", "keine angabe", "test"]

def _pos(rng):
    o = rng.choice(POS_OPENERS); a = rng.sample(POS, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "positive"
def _neg(rng):
    o = rng.choice(NEG_OPENERS); a = rng.sample(NEG, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "negative"
def _mixed(rng):
    p = rng.choice(POS); n = rng.choice(NEG)
    conn = rng.choice([" - aber ", ", allerdings ", ". Leider ", ", jedoch "])
    return f"{p[0].upper()+p[1:]}{conn}{n}.", "mixed"
def _neutral(rng):
    a = rng.choice(POS + NEG)
    return rng.choice(NEU_TEMPLATES).format(a=a, a_cap=a[0].upper()+a[1:]), "neutral"
def _rating_for(truth, rng):
    return rng.choice({"positive":[4,5,5],"negative":[1,1,2],"mixed":[2,3,4],
                       "neutral":[3,3,4],"fake":[5,5]}.get(truth,[1,3,5]))

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:   text, truth = _pos(rng)
        elif r < 0.60: text, truth = _neg(rng)
        elif r < 0.72: text, truth = _mixed(rng)
        elif r < 0.80: text, truth = _neutral(rng)
        elif r < 0.88: text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.93: text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98: text, truth = rng.choice(ENGLISH)
        else:          text, truth = rng.choice(JUNK), "junk"
        day = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day)).isoformat(),
            "product": PRODUCT,
            "rating": _rating_for(truth, rng),
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": text in SARCASTIC,
            "is_fake": text in FAKE,
        })
    if n > 20:
        for j, src in enumerate([5, 12, 30]):
            rows.append(dict(rows[src], review_id=f"R{n+j:04d}"))
    rng.shuffle(rows)
    return rows

In [ ]:
def classify(text):
    """Mock-Klassifikation (key-frei). Reicht fuer die Aggregation heute."""
    t = (text or "").lower()
    pos = sum(w in t for w in ["gut","top","super","toll","hervorragend","bequem",
                                "stabil","klasse","liebe","genial","perfekt","begeistert","gelungen"])
    neg = sum(w in t for w in ["schlecht","kaputt","teuer","stuerzt","stürzt","rauscht","billig",
                                "nervt","enttaeuscht","enttäuscht","leer","bricht","matschig","finger weg"])
    if pos and neg: return "mixed"
    if pos: return "positive"
    if neg: return "negative"
    return "neutral"

In [ ]:
reviews = generate(300)
analysiert = [{**r, 'sentiment': classify(r['text'])} for r in reviews]
print(len(analysiert), 'Bewertungen klassifiziert.')

## 1. Die Fragen einer Entscheiderin

Eine Gesamt-Prozentzahl (Sitzung 3) ist ein Anfang. Aber eine Produktmanagerin will mehr:

1. **Wird es besser oder schlechter?** → Sentiment über die Zeit
2. **Worüber wird am meisten geschimpft?** → Themen in negativen Bewertungen
3. **Wo stehen wir insgesamt?** → das Gesamtbild

Genau diese drei bauen wir jetzt.

## 2. Sentiment über die Zeit

Jede Bewertung hat ein Datum. Wir gruppieren nach **Monat** und zählen, wie viele positiv bzw. negativ waren.

In [ ]:
nach_monat = defaultdict(lambda: Counter())
for a in analysiert:
    monat = a['date'][:7]          # 'YYYY-MM'
    nach_monat[monat][a['sentiment']] += 1

monate = sorted(nach_monat)
print('Monat    pos  neg')
for m in monate:
    print(f"{m}   {nach_monat[m]['positive']:3}  {nach_monat[m]['negative']:3}")

## 3. Das Ganze als Chart

Zahlen in einer Tabelle sind schwer zu lesen. Ein **Liniendiagramm** macht den Verlauf sofort sichtbar:

In [ ]:
pos = [nach_monat[m]['positive'] for m in monate]
neg = [nach_monat[m]['negative'] for m in monate]

plt.figure(figsize=(9,4))
plt.plot(monate, pos, marker='o', color='#1f9d55', label='Positiv')
plt.plot(monate, neg, marker='o', color='#e3342f', label='Negativ')
plt.title('Sentiment im Zeitverlauf')
plt.ylabel('Anzahl Bewertungen')
plt.xticks(rotation=45, ha='right')
plt.legend(); plt.tight_layout(); plt.show()

> 📉 **Was seht ihr?** Gegen Jahresende steigen die negativen Bewertungen, die positiven fallen. Naheliegender Schluss: **„Das Produkt ist schlechter geworden.“** Haltet diesen Gedanken kurz fest.

## 4. Worüber wird geschimpft? (Themen × Sentiment)

Der Trend sagt *dass* es mehr Kritik gibt — nicht *woran* es liegt. Kreuzen wir Themen mit negativem Sentiment. (Einfache Stichwort-Themen, wie in Sitzung 5.)

In [ ]:
STICHWORT = {'Akku':['akku','laden','leer','batterie'], 'App':['app'],
             'Klang':['klang','sound','bass','rauscht'], 'Preis':['preis','teuer','euro'],
             'Komfort':['bequem','sitzt','ohr','passform'], 'Verbindung':['bluetooth','verbindung','bricht']}
def themen(t):
    t=t.lower(); return [k for k,ws in STICHWORT.items() if any(w in t for w in ws)]

beschwerden = Counter()
for a in analysiert:
    if a['sentiment']=='negative':
        for th in themen(a['text']): beschwerden[th]+=1
print('Top-Beschwerde-Themen:')
for th,c in beschwerden.most_common(): print(f'  {th:12} {c}')

## Der tiefere Punkt: Korrelation ist nicht Ursache

Zurück zu eurem Schluss von oben: *„Das Produkt ist schlechter geworden.“*

Der Chart zeigt: negative Bewertungen **korrelieren** mit der Zeit — sie nehmen zu. Aber sagt uns das **warum**? Nein. Genauso plausibel:

- Ein Influencer mit vielen unzufriedenen Followern hat spät gekauft.
- Ein Konkurrenzprodukt kam raus — die Erwartungen stiegen.
- Nach Weihnachten schreiben mehr Frustrierte (Geschenk-Retouren).
- **Oder** das Produkt *ist* tatsächlich schlechter geworden.

Die Daten allein können das **nicht** unterscheiden. Ein Trend ist ein *Hinweis*, keine *Erklärung*.

> 💡 **Kernpunkt:** Aggregierte Zahlen zeigen **Muster**, nicht **Ursachen**. Wer aus einem Trend vorschnell eine Ursache ableitet, trifft falsche Entscheidungen. Die Zahl ist der Anfang der Frage — nicht die Antwort.

## 5. Eure Aufgabe

Beantwortet eine **Manager-Frage** mit den Daten — und formuliert dazu, was ihr aus den Zahlen *nicht* schließen könnt.

In [ ]:
# Beispiel-Frage: Welcher Monat hatte den höchsten Negativ-Anteil?
for m in monate:
    c = nach_monat[m]
    gesamt = sum(c.values())
    anteil = round(100*c['negative']/gesamt) if gesamt else 0
    print(f'{m}: {anteil}% negativ')

> ✏️ **Eure Aufgabe:** Sucht euch eine eigene Frage — z. B. *„In welchem Monat war der Akku das häufigste Beschwerde-Thema?“* — und beantwortet sie mit Code. Schreibt **einen Satz** dazu, welche *Ursache* die Zahl **nicht** beweist.

## 6. Geschafft — und Ausblick

Ihr habt die analytische Ebene: Trends über die Zeit, Beschwerde-Themen, Manager-Fragen beantwortet — und gelernt, Muster nicht mit Ursachen zu verwechseln.

**Nächste Woche (25.11):** der **Meilenstein** — alles zusammensetzen zum fertigen *Review-Radar*-Dashboard. Das Werkzeug, das ihr in Sitzung 1 gesehen habt, gebaut von euch.